# Final Project: Single-Camera Bird's Eye View Scene Reconstruction with Object Detection

Computer Vision (CSE 559a)
Name: Irene Williams

Bird's Eye View (BEV) systems are increasingly used in modern vehicles to provide drivers with greater situational awareness for tasks such as parking, turning, and switching lanes. The goals of this project are to 1) successfully generate a panoramic image and convert it into a bird's eye view and 2) to integrate object detection and display detected objects on the final map.

In [1]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

## Setup

Here we define a few helper functions that will be used throughout the notebook. These functions handle loading images from the data folder, displaying images in Jupyter, and saving outputs.

In [2]:
def load_images(folder):
    extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    paths = []

    for ext in extensions:
        paths.extend(glob.glob(os.path.join(folder, ext)))
    paths = sorted(paths)

    images = []
    for path in paths:
        img = cv2.imread(path)
        if img is not None:
            images.append((path, img))

    return images

def display_image(img, title="", figsize=(10, 6)):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis("off")
    plt.show()

def save_image(path, img):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, img)

## Load Scene Images

In this section, we load the input images for one scene. These images should come from the same location, have substantial overlap, and be captured while rotating the camera gradually around the scene. This overlap is necessary for panorama stitching, since the algorithm relies on shared visual features between adjacent images.

In [3]:
scene_folder = "data/scene1"
images = load_images(scene_folder)

print(f"Loaded {len(images)} images from {scene_folder}")

Loaded 0 images from data/scene1


## Visualize Input Images

Before applying any preprocessing, we will visualize the raw images for a scene. This helps verify that the images were loaded correctly, that the images are in the expected order, that the scene has enough overlap for panorama stitching, and that lighting conditions are reasonably consistent.

In [4]:
if len(images) == 0:
    print("No images found.")
else:
    num_to_show = min(4, len(images))
    for path, img in images[:num_to_show]:
        display_image(img, title=os.path.basename(path), figsize=(8, 5))

No images found.


## Panorama Stitching

The first major goal in this project is panorama stitching, where multiple overlapping images are combined into a single wide-field representation. Panorama stitiching involves detecting keypoints in each image, matching features across image pairs. and warping/blending images into a common frame. We use OpenCV’s built in stitching pipeline as a baseline implementation of these steps.

In [5]:
pano = None

if len(images) < 2:
    print("Need at least 2 images for stitching.")
else:
    imgs = [img for _, img in images]
    
    stitcher = cv2.Stitcher_create()
    status, pano = stitcher.stitch(imgs)
    
    print("Status:", status)

Need at least 2 images for stitching.


In [6]:
if pano is None:
    print("No panorama to show.")
else:
    display_image(pano, title="Panorama", figsize=(14, 6))
    save_image("outputs/pano.jpg", pano)

No panorama to show.


## Bird's Eye View (Perspective Transform)

To approximate a top-down view of the scene, we apply a perspective transformation to the stitched panorama.

This transformation maps a selected quadrilateral region in the image to a rectangular output, effectively simulating a change in viewpoint. For this transformation to occur, we made the following assumptions: 

- the relevant portion of the scene lies on a planar surface,
- selected points correspond to ground locations,
- projective geometry can approximate the transformation between views

The transformation is defined by a homography matrix computed from four point correspondences.

In [7]:
# Placeholder points — update later once you see your panorama
src = np.float32([
    [300, 700],
    [1200, 700],
    [850, 350],
    [500, 350]
])

dst = np.float32([
    [0, 600],
    [800, 600],
    [800, 0],
    [0, 0]
])

if pano is None:
    print("No panorama — skipping BEV.")
    bev = None
else:
    H = cv2.getPerspectiveTransform(src, dst)
    bev = cv2.warpPerspective(pano, H, (800, 600))

No panorama — skipping BEV.


In [8]:
if bev is None:
    print("No BEV image.")
else:
    display_image(bev, title="Bird's Eye View", figsize=(8, 6))
    save_image("outputs/bev.jpg", bev)

No BEV image.


## Object Detection (YOLO)

To incorporate semantic understanding, we use YOLOv8, a pre-trained object detection model. YOLOv8 can identify relevant scene elements such as cars or pedestrians. We first test object detection on a single image to verify that the model loads correctly, inference runs successfully and detected objects are reasonable for the given scene.

In [9]:
model = YOLO("yolov8n.pt")
print("YOLO loaded")

YOLO loaded


In [10]:
det_img = None

if model is None:
    print("YOLO not available")
elif len(images) == 0:
    print("No images available")
else:
    path, _ = images[0]
    results = model(path)
    det_img = results[0].plot()

No images available


In [11]:
if det_img is None:
    print("No detection result")
else:
    display_image(det_img, title="Detection", figsize=(8, 6))
    save_image("outputs/detection.jpg", det_img)

No detection result


After validating detection on a single image, we apply the model to all images in the scene. This step ensures consistent performance across different viewpoints. The results are saved for later use and comparison.

In [12]:
if model is None or len(images) == 0:
    print("Skipping batch detection")
else:
    for path, _ in images:
        results = model(path)
        annotated = results[0].plot()

        name = os.path.basename(path)
        save_image(f"outputs/detections/{name}", annotated)

    print("Saved all detections")

Skipping batch detection
